# Machine Learning in Computational Biology - Assignment #2 - Spring 2026

## Imports

In [16]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path
import sys
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

project_dir = Path.cwd().parent
utils_dir = project_dir / 'src'

if str(utils_dir) not in sys.path:
    sys.path.append(str(utils_dir))

import functions
import rnCV_class

warnings.filterwarnings("ignore", category=UserWarning)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
dataset_raw = functions.load_data('students_dataset.csv')

X = dataset_raw.drop('num', axis=1)
y = dataset_raw['num']

print(f"Dataset shape : {dataset_raw.shape}")
print(f"Class counts : {y.value_counts().to_dict()}")

Dataset shape : (242, 14)
Class counts : {0: 131, 1: 111}


## Task 3 - Nested Cross-Validation Implementation

### Task 3.2 - Assess Generalization Performance and Select the Winner Algorithm

In [3]:
ALGORITHMS = [
    ("LR_ElasticNet", LogisticRegression(), functions.lr_space),
    ("GNB", GaussianNB(), functions.gnb_space),
    ("LDA", LinearDiscriminantAnalysis(), functions.lda_space),
    ("RandomForest", RandomForestClassifier(random_state=42), functions.rf_space),
    ("LightGBM", LGBMClassifier(random_state=42), functions.lgbm_space),
    ("XGBoost", XGBClassifier(random_state=42), functions.xgb_space),
    ("CatBoost", CatBoostClassifier(random_seed=42, verbose=0), functions.catboost_space),
]

print(f"{len(ALGORITHMS)} classifiers registered:")
print([name for name, _, _ in ALGORITHMS])

7 classifiers registered:
['LR_ElasticNet', 'GNB', 'LDA', 'RandomForest', 'LightGBM', 'XGBoost', 'CatBoost']


In [4]:
# Initialize rnCV class with desired parameters
rncv = rnCV_class.HeartDiseasernCV(n_rounds=10, n_outer=5, n_inner=3)

print(f"n_rounds : {rncv.n_rounds}")
print(f"n_outer  : {rncv.n_outer}  (evaluation folds per round)")
print(f"n_inner  : {rncv.n_inner}  (hyperparameter-tuning folds)")
print(f"Total evaluation points per algorithm : {rncv.n_rounds * rncv.n_outer}")
print("  (= 10 rounds x 5 outer folds)")

n_rounds : 10
n_outer  : 5  (evaluation folds per round)
n_inner  : 3  (hyperparameter-tuning folds)
Total evaluation points per algorithm : 50
  (= 10 rounds x 5 outer folds)


In [8]:
baseline_results = {}

for name, clf, space_fn in ALGORITHMS:
    print(f"Baseline {name} ...", end=" ", flush=True)
    baseline_results[name] = rncv.run_assessment(
        X, y,
        estimator_name = name,
        estimator = clf,
        param_space_func = space_fn,
        perform_fs = False,
        tune = False, # inner loop disabled
    )
    med = baseline_results[name]["mcc"].median()
    print(f"done | Median MCC = {med:.3f}")

print("\nBaseline complete.")

Baseline LR_ElasticNet ... done | Median MCC = 0.631
Baseline GNB ... done | Median MCC = 0.477
Baseline LDA ... done | Median MCC = 0.631
Baseline RandomForest ... done | Median MCC = 0.583
Baseline LightGBM ... [LightGBM] [Info] Number of positive: 88, number of negative: 105
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000133 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 215
[LightGBM] [Info] Number of data points in the train set: 193, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.455959 -> initscore=-0.176624
[LightGBM] [Info] Start training from score -0.176624
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [

In [9]:
rncv_results = {}

for name, clf, space_fn in ALGORITHMS:
    print(f"rnCV {name} ...", end=" ", flush=True)
    rncv_results[name] = rncv.run_assessment(
        X, y,
        estimator_name = name,
        estimator = clf,
        param_space_func = space_fn,
        perform_fs = False,
        tune = True, # inner Optuna loop active
    )
    med = rncv_results[name]["mcc"].median()
    print(f"done | Median MCC = {med:.3f}")

print("\nrnCV complete.")

rnCV LR_ElasticNet ... 

[I 2026-05-13 22:19:20,122] A new study created in memory with name: no-name-9941fafa-c837-40d7-9b33-388270fe5996


[I 2026-05-13 22:19:20,226] Trial 0 finished with value: 0.674264007597341 and parameters: {'C': 0.05547119471592124, 'l1_ratio': 0.7151893663724195}. Best is trial 0 with value: 0.674264007597341.
[I 2026-05-13 22:19:20,314] Trial 1 finished with value: 0.7915315846350329 and parameters: {'C': 0.10323260351976568, 'l1_ratio': 0.5448831829968969}. Best is trial 1 with value: 0.7915315846350329.
[I 2026-05-13 22:19:20,381] Trial 2 finished with value: 0.0 and parameters: {'C': 0.013130280280658591, 'l1_ratio': 0.6458941130666561}. Best is trial 1 with value: 0.7915315846350329.
[I 2026-05-13 22:19:20,447] Trial 3 finished with value: 0.0 and parameters: {'C': 0.015414734761917091, 'l1_ratio': 0.8917730007820798}. Best is trial 1 with value: 0.7915315846350329.
[I 2026-05-13 22:19:20,587] Trial 4 finished with value: 0.8115975422427035 and parameters: {'C': 6.581332043291798, 'l1_ratio': 0.3834415188257777}. Best is trial 4 with value: 0.8115975422427035.
[I 2026-05-13 22:19:20,681] Tria

done | Median MCC = 0.627
rnCV GNB ... 

[I 2026-05-13 22:22:30,488] A new study created in memory with name: no-name-f5e65163-a8b0-40e6-9bd3-b9fcbb089002
[I 2026-05-13 22:22:30,595] Trial 0 finished with value: 0.7645536206255846 and parameters: {'var_smoothing': 5.5471194715921156e-09}. Best is trial 0 with value: 0.7645536206255846.
[I 2026-05-13 22:22:30,687] Trial 1 finished with value: 0.7723169416881385 and parameters: {'var_smoothing': 3.766576841599302e-08}. Best is trial 1 with value: 0.7723169416881385.
[I 2026-05-13 22:22:30,774] Trial 2 finished with value: 0.7683899377611345 and parameters: {'var_smoothing': 1.032326035197657e-08}. Best is trial 1 with value: 0.7723169416881385.
[I 2026-05-13 22:22:30,860] Trial 3 finished with value: 0.7645536206255846 and parameters: {'var_smoothing': 5.30170934757682e-09}. Best is trial 1 with value: 0.7723169416881385.
[I 2026-05-13 22:22:30,952] Trial 4 finished with value: 0.7699631197059698 and parameters: {'var_smoothing': 1.3130280280658582e-09}. Best is trial 1 with va

...

done | Median MCC = 0.606
rnCV XGBoost ... 

[I 2026-05-13 23:12:55,100] A new study created in memory with name: no-name-75654608-abdf-495d-98af-d7b86433c6da
[I 2026-05-13 23:12:56,685] Trial 0 finished with value: 0.776661502946358 and parameters: {'n_estimators': 187, 'max_depth': 7, 'learning_rate': 0.07768811685516967, 'subsample': 0.8179532731987588, 'colsample_bytree': 0.7694619197355619, 'reg_alpha': 0.3833332156156664, 'reg_lambda': 0.0562793204741517}. Best is trial 0 with value: 0.776661502946358.
[I 2026-05-13 23:12:58,380] Trial 1 finished with value: 0.7474069197571501 and parameters: {'n_estimators': 273, 'max_depth': 8, 'learning_rate': 0.036845938035010996, 'subsample': 0.9166900152330658, 'colsample_bytree': 0.8115579679011617, 'reg_alpha': 0.18714500686240676, 'reg_lambda': 5.039489598671215}. Best is trial 0 with value: 0.776661502946358.
[I 2026-05-13 23:12:58,619] Trial 2 finished with value: 0.6552929828791898 and parameters: {'n_estimators': 67, 'max_depth': 3, 'learning_rate': 0.010711863369746433, 'subsa

done | Median MCC = 0.589
rnCV CatBoost ... 

[I 2026-05-13 23:27:39,480] A new study created in memory with name: no-name-4de69a57-5f9e-44f0-b74e-c3ae9f5d0682
[I 2026-05-13 23:27:41,528] Trial 0 finished with value: 0.7638085742771685 and parameters: {'iterations': 187, 'depth': 7, 'learning_rate': 0.07768811685516967, 'l2_leaf_reg': 0.15119336467641012}. Best is trial 0 with value: 0.7638085742771685.
[I 2026-05-13 23:27:42,600] Trial 1 finished with value: 0.7640999493824286 and parameters: {'iterations': 156, 'depth': 6, 'learning_rate': 0.04429649570464297, 'l2_leaf_reg': 3.6905577292137624}. Best is trial 1 with value: 0.7640999493824286.
[I 2026-05-13 23:27:43,943] Trial 2 finished with value: 0.7526056965088653 and parameters: {'iterations': 291, 'depth': 5, 'learning_rate': 0.1477317633429356, 'l2_leaf_reg': 0.13049073550362397}. Best is trial 1 with value: 0.7640999493824286.
[I 2026-05-13 23:27:47,101] Trial 3 finished with value: 0.7703887510339124 and parameters: {'iterations': 192, 'depth': 8, 'learning_rate': 0.0127

done | Median MCC = 0.623

rnCV complete.


In [ ]:
METRICS = ["mcc", "auc", "prauc", "ba", "f1", "recall", "specificity", "precision"]

rows = []
for name, df_res in rncv_results.items():
    for metric in METRICS:
        med = df_res[metric].median()
        lo, hi = functions.median_ci(df_res[metric])
        rows.append({
            "Algorithm" : name,
            "Metric"    : metric.upper(),
            "Median"    : round(med, 3),
            "95% CI"    : f"[{lo:.3f}, {hi:.3f}]",
        })

comparison_df = pd.DataFrame(rows)

# Wide pivot: algorithms as rows, metrics as columns (sorted by MCC)
pivot = (
    comparison_df
    .pivot(index="Algorithm", columns="Metric", values="Median")
    [["MCC", "AUC", "PRAUC", "BA", "F1", "RECALL", "SPECIFICITY", "PRECISION"]]
    .sort_values("MCC", ascending=False)
)
print("Median performance across 50 rnCV evaluation points (sorted by MCC):")
display(pivot)

print("\n95% Bootstrap CIs:")
display(
    comparison_df
    .pivot(index="Algorithm", columns="Metric", values="95% CI")
    .reindex(pivot.index)
)

Median performance across 50 rnCV evaluation points (sorted by MCC):


Metric,MCC,AUC,PRAUC,BA,F1,RECALL,SPECIFICITY,PRECISION
Algorithm,,,,,,,,
LDA,0.656,0.897,0.892,0.819,0.800,0.773,0.885,0.834
LR_ElasticNet,0.627,0.894,0.893,0.808,0.782,0.773,0.885,0.841
CatBoost,0.623,0.887,0.881,0.805,0.780,0.756,0.846,0.810
LightGBM,0.606,0.880,0.873,0.798,0.776,0.773,0.846,0.810
RandomForest,0.592,0.896,0.892,0.795,0.777,0.756,0.846,0.800
XGBoost,0.589,0.879,0.881,0.793,0.773,0.773,0.846,0.800
GNB,0.532,0.877,0.868,0.760,0.760,0.867,0.654,0.667



95% Bootstrap CIs:


Metric,AUC,BA,F1,MCC,PRAUC,PRECISION,RECALL,SPECIFICITY
Algorithm,,,,,,,,
LDA,"[0.886, 0.903]","[0.803, 0.834]","[0.780, 0.818]","[0.623, 0.673]","[0.878, 0.904]","[0.800, 0.854]","[0.727, 0.783]","[0.846, 0.885]"
LR_ElasticNet,"[0.885, 0.908]","[0.790, 0.832]","[0.769, 0.817]","[0.599, 0.669]","[0.877, 0.903]","[0.809, 0.850]","[0.727, 0.818]","[0.846, 0.885]"
CatBoost,"[0.874, 0.897]","[0.776, 0.813]","[0.757, 0.800]","[0.551, 0.644]","[0.873, 0.896]","[0.762, 0.850]","[0.727, 0.773]","[0.808, 0.885]"
LightGBM,"[0.848, 0.890]","[0.778, 0.812]","[0.762, 0.791]","[0.557, 0.632]","[0.857, 0.894]","[0.762, 0.833]","[0.733, 0.795]","[0.808, 0.885]"
RandomForest,"[0.878, 0.913]","[0.779, 0.818]","[0.756, 0.800]","[0.560, 0.635]","[0.877, 0.908]","[0.771, 0.834]","[0.727, 0.773]","[0.808, 0.885]"
XGBoost,"[0.865, 0.895]","[0.781, 0.808]","[0.762, 0.791]","[0.568, 0.632]","[0.874, 0.894]","[0.762, 0.842]","[0.727, 0.773]","[0.808, 0.885]"
GNB,"[0.863, 0.883]","[0.740, 0.779]","[0.743, 0.774]","[0.493, 0.572]","[0.855, 0.879]","[0.645, 0.691]","[0.864, 0.909]","[0.577, 0.667]"


In [19]:
WINNER_NAME = pivot.index[0]

winner_name, winner_clf, winner_space = next(
    (n, c, s) for n, c, s in ALGORITHMS if n == WINNER_NAME
)

print(f"Winner : {winner_name}")
display(rncv_results[winner_name][METRICS].describe().T[["mean", "50%", "std"]])

Winner : LDA


,mean,50%,std
mcc,0.639722,0.655575,0.110891
auc,0.895997,0.897423,0.039105
prauc,0.891966,0.891797,0.041050
ba,0.815333,0.819412,0.055018
f1,0.795152,0.800000,0.062634
recall,0.768300,0.772727,0.084731
specificity,0.862365,0.884615,0.073643
precision,0.831701,0.834096,0.079131


## Task 4 - Feature Selection

In [21]:
print(f"Running rnCV + feature selection for {winner_name} ...")

fs_results = rncv.run_assessment(
    X, y,
    estimator_name = winner_name,
    estimator = winner_clf,
    param_space_func = winner_space,
    perform_fs = True, # RFECV selector inserted into the pipeline
    tune = True, # inner hyperparameter tuning remains active
)

print("\nFeature-selection rnCV complete.")

[I 2026-05-14 00:07:18,846] A new study created in memory with name: no-name-878584cd-2028-41ef-9def-fcf2e133b112


[I 2026-05-14 00:07:18,911] Trial 0 finished with value: 0.8342472342472341 and parameters: {'shrinkage': None}. Best is trial 0 with value: 0.8342472342472341.
[I 2026-05-14 00:07:18,989] Trial 1 finished with value: 0.8145021645021645 and parameters: {'shrinkage': 'auto'}. Best is trial 0 with value: 0.8342472342472341.


Running rnCV + feature selection for LDA ...


[I 2026-05-14 00:07:19,058] Trial 2 finished with value: 0.8342472342472341 and parameters: {'shrinkage': None}. Best is trial 0 with value: 0.8342472342472341.
[I 2026-05-14 00:07:19,135] Trial 3 finished with value: 0.8342472342472341 and parameters: {'shrinkage': None}. Best is trial 0 with value: 0.8342472342472341.
[I 2026-05-14 00:07:19,205] Trial 4 finished with value: 0.8145021645021645 and parameters: {'shrinkage': 'auto'}. Best is trial 0 with value: 0.8342472342472341.
[I 2026-05-14 00:07:19,275] Trial 5 finished with value: 0.8145021645021645 and parameters: {'shrinkage': 'auto'}. Best is trial 0 with value: 0.8342472342472341.
[I 2026-05-14 00:07:19,342] Trial 6 finished with value: 0.8342472342472341 and parameters: {'shrinkage': None}. Best is trial 0 with value: 0.8342472342472341.
[I 2026-05-14 00:07:19,402] Trial 7 finished with value: 0.8342472342472341 and parameters: {'shrinkage': None}. Best is trial 0 with value: 0.8342472342472341.
[I 2026-05-14 00:07:19,476] Tr

ValueError: operands could not be broadcast together with shapes (22,) (21,) (22,) 

In [ ]:
full_res = rncv_results[winner_name] # all features, tuned
fs_res = fs_results # selected features, tuned

compare_rows = []
for metric in METRICS:
    med_full, med_fs = full_res[metric].median(), fs_res[metric].median()
    lo_full, hi_full = functions.median_ci(full_res[metric])
    lo_fs, hi_fs = functions.median_ci(fs_res[metric])
    compare_rows.append({
        "Metric"        : metric.upper(),
        "Full Median"   : round(med_full, 3),
        "Full 95% CI"   : f"[{lo_full:.3f}, {hi_full:.3f}]",
        "FS Median"     : round(med_fs, 3),
        "FS 95% CI"     : f"[{lo_fs:.3f}, {hi_fs:.3f}]",
        "Delta (FS-Full)": round(med_fs - med_full, 3),
    })

compare_df = pd.DataFrame(compare_rows).set_index("Metric")
print(f"{winner_name}: Full features vs. RFECV-selected subset")
display(compare_df)